# Part 2 · Dataset and preprocessing

**Dataset:** *Fake News Detection Dataset* by Mahdi Mashayekhi (Kaggle, MIT licence), 20,000 articles, 7 columns.

This notebook produces everything for report section **b) Dataset and Preprocessing**:

1. **Description:** domain, structure, size, source.
2. **Initial observations and statistics:** missing values, duplicates, class balance, metadata distributions,
   text lengths, vocabulary, and whether any column carries information about the label.
3. **Preprocessing:** missing-value handling, de-duplication, text cleaning, tokenization, stopword removal,
   stemming and lemmatization (for TF‑IDF), plus WordPiece tokenization (for DistilBERT).
4. **Examples and effect:** one article shown after every step, and the change in tokens and vocabulary per step.
5. **Splits:** stratified 70 / 15 / 15 train / validation / test, saved to Drive for the models in section d.

Runs on **CPU** in about 5 minutes. Everything is saved to `MyDrive/NLP_part2/`.

### ▶ Keep this tab awake (tap play once)

Phone browsers suspend background tabs, which can freeze the notebook's progress display or drop the
connection while long cells run. The player below loops a **near-silent** 5‑second sound. Tap ▶ once
after starting *Run all*; it keeps playing in your browser while the other cells run.
It does **not** extend Colab's own runtime limits, so keep the phone charging and Colab open.

In [ ]:
import io, wave, base64
import numpy as np
from IPython.display import HTML, display

_sr = 8000
_noise = (np.random.default_rng(0).standard_normal(_sr * 5) * 20).astype(np.int16)   # ~-64 dBFS: inaudible
_buf = io.BytesIO()
with wave.open(_buf, "wb") as _w:
    _w.setnchannels(1); _w.setsampwidth(2); _w.setframerate(_sr); _w.writeframes(_noise.tobytes())
display(HTML(f'<audio controls loop autoplay src="data:audio/wav;base64,{base64.b64encode(_buf.getvalue()).decode()}"></audio>'
             '<div style="font-size:12px;color:gray">Keep-alive player: tap ▶ if it is not already playing.</div>'))

## 0 · Configuration

In [ ]:
SEED = 42
SPLIT_FRACS = (0.70, 0.15, 0.15)
KAGGLE_DATASET = "mahdimashayekhi/fake-news-detection-dataset"
CSV_NAME = "fake_news_dataset.csv"
PROJECT_DIRNAME = "NLP_part2"
BERT_MODEL = "distilbert-base-uncased"
BERT_MAX_LEN = 512

## 1 · Setup

In [ ]:
import os, sys, re, json, glob, shutil
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = f"/content/drive/MyDrive/{PROJECT_DIRNAME}"
else:
    BASE_DIR = os.environ.get("P2_BASE_DIR", os.path.abspath(f"./{PROJECT_DIRNAME}"))
DIRS = {k: os.path.join(BASE_DIR, k) for k in ["data", "results", "figures"]}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)
pd.set_option("display.max_colwidth", 120)
print("Outputs ->", BASE_DIR)

## 2 · Load the dataset

Sources, tried in order:
1. a copy already on Drive (`NLP_part2/data/fake_news_dataset.csv`),
2. a direct download from Kaggle with `kagglehub` (public dataset, no login needed).

If both fail, download `fake_news_dataset.csv` from the Kaggle page on your phone and upload it to
`MyDrive/NLP_part2/data/`, then run this notebook again.

In [ ]:
CSV_PATH = os.path.join(DIRS["data"], CSV_NAME)
if not os.path.exists(CSV_PATH):
    try:
        import kagglehub
        src_dir = kagglehub.dataset_download(KAGGLE_DATASET)
        shutil.copy(glob.glob(os.path.join(src_dir, "**", CSV_NAME), recursive=True)[0], CSV_PATH)
        print("Downloaded from Kaggle and saved to Drive")
    except Exception as e:
        raise SystemExit(f"Could not download the dataset ({e}). Upload {CSV_NAME} to {DIRS['data']} and re-run.")
raw = pd.read_csv(CSV_PATH)
print(f"Loaded {CSV_PATH}: {raw.shape[0]:,} rows x {raw.shape[1]} columns")

## 3 · Dataset description

In [ ]:
print("Column types:")
print(raw.dtypes.to_string(), "\n")
display(raw.head(5))

In [ ]:
COLUMN_INFO = {
    "title": "Headline of the article",
    "text": "Body of the article",
    "date": "Publication date",
    "source": "Publishing organisation",
    "author": "Author's name",
    "category": "Topic of the article",
    "label": "Target: real or fake",
}
desc = pd.DataFrame({
    "description": pd.Series(COLUMN_INFO),
    "dtype": raw.dtypes.astype(str),
    "non-null": raw.notna().sum(),
    "missing %": (100 * raw.isna().mean()).round(2),
    "unique values": raw.nunique(),
    "example": raw.iloc[0].astype(str).str.slice(0, 60),
}).loc[raw.columns]
desc

## 4 · Initial observations and statistics

### 4.1 · Missing values, duplicates, class balance

In [ ]:
stats = {}
stats["rows"], stats["columns"] = raw.shape
stats["missing_by_column"] = raw.isna().sum().to_dict()
stats["exact_duplicate_rows"] = int(raw.duplicated().sum())
stats["duplicate_titles"] = int(raw["title"].duplicated().sum())
stats["duplicate_texts"] = int(raw["text"].duplicated().sum())
stats["label_counts"] = raw["label"].value_counts().to_dict()

print("Missing values per column:\n", raw.isna().sum().to_string(), "\n")
print(f"Exact duplicate rows: {stats['exact_duplicate_rows']}, duplicate titles: {stats['duplicate_titles']}, "
      f"duplicate texts: {stats['duplicate_texts']}")
print("\nClass balance:\n", raw["label"].value_counts().to_string())
print((100 * raw["label"].value_counts(normalize=True)).round(2).to_string())

### 4.2 · Metadata: category, source, author, date

In [ ]:
df0 = raw.copy()
df0["date"] = pd.to_datetime(df0["date"], errors="coerce")
print("Date range:", df0["date"].min().date(), "to", df0["date"].max().date())
print("Unique authors:", df0["author"].nunique(), "| unique sources:", df0["source"].nunique(),
      "| unique categories:", df0["category"].nunique(), "\n")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col in zip(axes, ["category", "source"]):
    ct = pd.crosstab(df0[col].fillna("(missing)"), df0["label"])
    ct.loc[ct.sum(axis=1).sort_values(ascending=False).index].plot.bar(ax=ax, rot=45)
    ax.set_title(f"Articles per {col} and label"); ax.set_xlabel(""); ax.set_ylabel("articles")
fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], "p2b_category_source.png"), dpi=150); plt.show()

print("Share of fake articles per category (%):")
print((100 * pd.crosstab(df0["category"], df0["label"], normalize="index")).round(1).to_string())

### 4.3 · Text length and vocabulary

In [ ]:
df0["title_words"] = df0["title"].fillna("").str.split().str.len()
df0["text_words"] = df0["text"].fillna("").str.split().str.len()
df0["text_chars"] = df0["text"].fillna("").str.len()
df0["text_sentences"] = df0["text"].fillna("").str.count(r"[.!?]+")

length_table = df0.groupby("label")[["title_words", "text_words", "text_chars", "text_sentences"]].describe().T.round(1)
display(length_table)

fig, ax = plt.subplots(figsize=(7, 3.5))
for lab in sorted(df0["label"].dropna().unique()):
    ax.hist(df0.loc[df0["label"] == lab, "text_words"], bins=40, alpha=0.6, label=lab)
ax.set_xlabel("words in article body"); ax.set_ylabel("articles"); ax.legend(); ax.set_title("Article length by label")
fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], "p2b_length_hist.png"), dpi=150); plt.show()

all_words = Counter(w for t in df0["text"].fillna("").str.lower() for w in re.findall(r"[a-z']+", t))
stats["raw_vocabulary_size"] = len(all_words)
stats["raw_total_words"] = int(sum(all_words.values()))
print(f"Vocabulary (lower-cased words): {len(all_words):,} distinct words, {sum(all_words.values()):,} words in total")
print("Most common words:", ", ".join(f"{w} ({c:,})" for w, c in all_words.most_common(20)))

### 4.4 · Does any column carry information about the label?

Before modelling we test, column by column, whether the label depends on it. For each test,
a **large p-value** means real and fake articles look the same on that column.

* category and source vs. label: chi-square test of independence
* text length and date vs. label: Mann–Whitney U test
* individual words vs. label: chi-square test for each of the 2,000 most frequent words, counting how many are
  significant after Bonferroni correction (on a dataset with genuine word–label patterns, many would be)

In [ ]:
from scipy.stats import chi2_contingency, mannwhitneyu
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import chi2

y_fake = (df0["label"].astype(str).str.lower() == "fake").astype(int)
tests = []
for col in ["category", "source"]:
    ct = pd.crosstab(df0[col].fillna("(missing)"), y_fake)
    tests.append({"column": col, "test": "chi-square", "p_value": chi2_contingency(ct)[1]})
for col in ["text_words", "title_words"]:
    tests.append({"column": col, "test": "Mann-Whitney U",
                  "p_value": mannwhitneyu(df0.loc[y_fake == 1, col], df0.loc[y_fake == 0, col]).pvalue})
d_num = df0["date"].map(lambda d: d.toordinal() if pd.notna(d) else np.nan)
tests.append({"column": "date", "test": "Mann-Whitney U",
              "p_value": mannwhitneyu(d_num[y_fake == 1].dropna(), d_num[y_fake == 0].dropna()).pvalue})

cv = CountVectorizer(max_features=2000, lowercase=True)
Xw = cv.fit_transform(df0["text"].fillna(""))
_, p_words = chi2(Xw, y_fake)
n_sig = int((p_words < 0.05 / len(p_words)).sum())
tests.append({"column": f"{len(p_words)} most frequent words", "test": "chi-square per word (Bonferroni)",
              "p_value": min(1.0, float(np.min(p_words)) * len(p_words))})
tests_df = pd.DataFrame(tests)
tests_df["label-related at 5%?"] = np.where(tests_df["p_value"] < 0.05, "yes", "no")
display(tests_df)
print(f"Words significantly associated with the label after correction: {n_sig} of {len(p_words)}")
stats["label_dependence_tests"] = tests_df.to_dict(orient="records")
stats["significant_words_bonferroni"] = n_sig

In [ ]:
print("Two random articles of each label:\n")
for lab in sorted(df0["label"].dropna().unique()):
    for _, r in df0[df0["label"] == lab].sample(2, random_state=SEED).iterrows():
        print(f"[{lab}] {r['title']}  |  {r['source']} · {r['author']} · {r['category']} · {str(r['date'])[:10]}")
        print("   ", str(r["text"])[:300], "...\n")

## 5 · Preprocessing

Two pipelines, because the two models need different input:

| Step | TF‑IDF + LR (classical) | DistilBERT (transformer) |
|---|---|---|
| Missing `source` / `author` | filled with `"Unknown"` | same |
| Duplicates, empty texts | removed | same |
| Label | `real` → 0, `fake` → 1 | same |
| Model input | `title + text` | `title` and `text` as a sentence pair |
| Cleaning | lower-case; remove URLs, HTML, digits, punctuation | none (the tokenizer lower-cases) |
| Tokenization | word tokens (regex) | WordPiece sub-word tokens |
| Stopword removal | yes (scikit-learn English list, 318 words) | no: the model needs function words for context |
| Stemming / lemmatization | Porter stemming (lemmatization compared) | no: WordPiece already splits rare words into pieces |

### 5.1 · Cleaning the table

In [ ]:
log = [("loaded", len(raw))]
df = raw.copy()
df["source"] = df["source"].fillna("Unknown")
df["author"] = df["author"].fillna("Unknown")
df = df.dropna(subset=["title", "text", "label"]);                    log.append(("title, text and label present", len(df)))
df["title"] = df["title"].astype(str).str.strip()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0];                                    log.append(("non-empty text", len(df)))
df = df.drop_duplicates(subset=["title", "text"]);                    log.append(("duplicates removed", len(df)))
df["fake"] = (df["label"].str.lower() == "fake").astype(int)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.reset_index(drop=True)
df.insert(0, "id", np.arange(len(df)))

clean_log = pd.DataFrame(log, columns=["step", "rows remaining"])
clean_log["removed"] = (-clean_log["rows remaining"].diff()).fillna(0).astype(int)
display(clean_log)
print("Missing values after filling:", int(df[["source", "author"]].isna().sum().sum()))

### 5.2 · Text pipeline for TF‑IDF

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.stem import PorterStemmer

STOPWORDS = frozenset(ENGLISH_STOP_WORDS)
stemmer = PorterStemmer()
TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)?")

try:      # WordNet lemmatizer: needs a small NLTK download; skipped if unavailable
    import nltk
    nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
    from nltk.stem import WordNetLemmatizer
    _lem = WordNetLemmatizer(); _lem.lemmatize("tests")
    lemmatize = _lem.lemmatize
except Exception as e:
    print("Lemmatizer unavailable, skipping it:", type(e).__name__)
    lemmatize = None

def clean(s):
    s = s.lower()
    s = re.sub(r"https?://\S+|www\.\S+", " ", s)      # URLs
    s = re.sub(r"<[^>]+>", " ", s)                    # HTML tags
    s = re.sub(r"[^a-z'\s]", " ", s)                  # digits and punctuation
    return re.sub(r"\s+", " ", s).strip()

def tokenize(s):
    return TOKEN_RE.findall(s)

def remove_stopwords(toks):
    return [t for t in toks if t not in STOPWORDS and len(t) > 1]

_stem_cache = {}
def stem(toks):
    return [_stem_cache.setdefault(t, stemmer.stem(t)) for t in toks]

_lem_cache = {}
def lemma(toks):
    return [_lem_cache.setdefault(t, lemmatize(t)) for t in toks]

def pipeline(title, text):
    steps = {"raw": f"{title}\n{text}"}
    steps["cleaned"] = clean(steps["raw"])
    steps["tokens"] = tokenize(steps["cleaned"])
    steps["no_stopwords"] = remove_stopwords(steps["tokens"])
    steps["stemmed"] = stem(steps["no_stopwords"])
    if lemmatize:
        steps["lemmatized"] = lemma(steps["no_stopwords"])
    return steps

# Run on every article
results = [pipeline(a, b) for a, b in zip(df["title"], df["text"])]
for key in ["tokens", "no_stopwords", "stemmed"] + (["lemmatized"] if lemmatize else []):
    df[key] = [r[key] for r in results]
df["text_clean"] = df["stemmed"].str.join(" ")          # final TF-IDF input
print("Done:", len(df), "articles processed")

### 5.3 · Example: one article through every step

In [ ]:
ex = df.iloc[0]
steps = pipeline(ex["title"], ex["text"])
show = lambda v: (" ".join(v) if isinstance(v, list) else v)[:400]
example_table = pd.DataFrame({"step": list(steps), "output (first 400 characters)": [show(v) for v in steps.values()],
                              "tokens": [len(v) if isinstance(v, list) else len(v.split()) for v in steps.values()]})
display(example_table)

print("Word-by-word, first 15 tokens:")
cols = {"token": steps["tokens"][:15]}
cols["stopword?"] = ["yes" if t in STOPWORDS else "" for t in cols["token"]]
cols["stem"] = [stemmer.stem(t) for t in cols["token"]]
if lemmatize:
    cols["lemma"] = [lemmatize(t) for t in cols["token"]]
display(pd.DataFrame(cols))

### 5.4 · Effect of each step on the whole dataset

In [ ]:
effect = []
for key, name in [("tokens", "tokenized"), ("no_stopwords", "+ stopwords removed"), ("stemmed", "+ Porter stemming")] + \
                 ([("lemmatized", "+ lemmatization (instead of stemming)")] if lemmatize else []):
    vocab = Counter(t for toks in df[key] for t in toks)
    effect.append({"step": name, "avg tokens per article": round(df[key].str.len().mean(), 1),
                   "vocabulary size": len(vocab), "total tokens": int(sum(vocab.values()))})
effect_df = pd.DataFrame(effect)
effect_df["vocab change vs tokenized"] = (100 * (effect_df["vocabulary size"] / effect_df["vocabulary size"].iloc[0] - 1)).round(1).astype(str) + "%"
display(effect_df)

before = Counter(t for toks in df["tokens"] for t in toks).most_common(15)
after = Counter(t for toks in df["stemmed"] for t in toks).most_common(15)
display(pd.DataFrame({"top words before": [f"{w} ({c:,})" for w, c in before],
                      "top words after stopwords + stemming": [f"{w} ({c:,})" for w, c in after]}))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(effect_df["step"], effect_df["vocabulary size"])
ax.set_ylabel("distinct tokens"); ax.set_title("Vocabulary size after each preprocessing step")
plt.setp(ax.get_xticklabels(), rotation=15, ha="right")
fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], "p2b_vocab_steps.png"), dpi=150); plt.show()

### 5.5 · Tokenization for DistilBERT

DistilBERT uses its own **WordPiece** tokenizer on the raw text: it lower-cases, splits punctuation, and breaks
words it does not know into sub-word pieces (marked `##`). Title and text are passed as a sentence pair.

In [ ]:
try:
    from transformers import AutoTokenizer
    bert_tok = AutoTokenizer.from_pretrained(BERT_MODEL)
    enc = bert_tok(ex["title"], ex["text"])
    print("First 40 WordPiece tokens of the example article:")
    print(bert_tok.convert_ids_to_tokens(enc["input_ids"])[:40], "\n")
    lens = np.array([len(i) for i in bert_tok(df["title"].tolist(), df["text"].tolist())["input_ids"]])
    stats["bert_tokens"] = {"mean": float(lens.mean()), "median": float(np.median(lens)), "max": int(lens.max()),
                            "share_over_512": float((lens > BERT_MAX_LEN).mean()), "share_over_256": float((lens > 256).mean())}
    print(f"WordPiece tokens per article: mean {lens.mean():.0f}, median {np.median(lens):.0f}, max {lens.max()}")
    print(f"Articles longer than 256 tokens: {100 * (lens > 256).mean():.1f}%  |  longer than 512: {100 * (lens > 512).mean():.1f}%")
except Exception as e:
    print("Skipped (tokenizer unavailable):", type(e).__name__, e)

### 5.6 · Is the preprocessing appropriate? A quick check

A preview of section d: a TF‑IDF + Logistic Regression model trained on the training split with each
preprocessing variant, scored on the validation split. On a dataset with real word–label patterns, these
variants usually differ by a point or two; here they show whether any of them finds a signal at all.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

tr_frac, va_frac, te_frac = SPLIT_FRACS
train_df, rest = train_test_split(df, train_size=tr_frac, stratify=df["fake"], random_state=SEED)
val_df, test_df = train_test_split(rest, test_size=te_frac / (va_frac + te_frac), stratify=rest["fake"], random_state=SEED)

variants = {"tokens only": "tokens", "+ stopwords removed": "no_stopwords", "+ stemming": "stemmed"}
if lemmatize:
    variants["+ lemmatization"] = "lemmatized"
ablation = []
for name, key in variants.items():
    vec = TfidfVectorizer(analyzer=lambda toks: toks, min_df=2)
    Xtr, Xva = vec.fit_transform(train_df[key]), vec.transform(val_df[key])
    pred = LogisticRegression(max_iter=2000).fit(Xtr, train_df["fake"]).predict(Xva)
    ablation.append({"preprocessing": name, "features": Xtr.shape[1], "val accuracy": round(accuracy_score(val_df["fake"], pred), 4),
                     "val macro-F1": round(f1_score(val_df["fake"], pred, average="macro"), 4)})
ablation_df = pd.DataFrame(ablation)
display(ablation_df)
print(f"Majority-class baseline: {max(val_df['fake'].mean(), 1 - val_df['fake'].mean()):.4f}")

## 6 · Save splits and summary

In [ ]:
keep = ["id", "title", "text", "date", "source", "author", "category", "label", "fake", "text_clean"]
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    part[keep].to_parquet(os.path.join(DIRS["data"], f"{name}.parquet"), index=False)
    print(f"{name}: {len(part):,} articles, {100 * part['fake'].mean():.1f}% fake")

summary = {
    "notebook": "p2_dataset_preprocessing",
    "dataset": KAGGLE_DATASET,
    "stats": stats,
    "cleaning_log": dict(log),
    "preprocessing_effect": effect_df.to_dict(orient="records"),
    "preprocessing_ablation_val": ablation_df.to_dict(orient="records"),
    "split_sizes": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
}
with open(os.path.join(DIRS["results"], "p2b_summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)
for name, t in [("description", desc), ("label_tests", tests_df), ("cleaning_log", clean_log),
                ("example", example_table), ("effect", effect_df), ("ablation", ablation_df)]:
    t.to_csv(os.path.join(DIRS["results"], f"p2b_{name}.csv"))
print("Saved results/p2b_summary.json, results/p2b_*.csv and figures/p2b_*.png")

**Done.** `MyDrive/NLP_part2/` now holds the cleaned splits (`data/{train,val,test}.parquet`), the summary and
tables for the report (`results/`), and the figures (`figures/`). Tell Claude it has finished so it can write
section b from these numbers.